# Notebook 02c — LLM-as-Classifier (re-measurement step)

**Purpose.** Re-score the cleaned 4,500-comment corpus using a Large Language Model (Claude or GPT-4) instead of the zero-shot DeBERTa classifier from Notebook 02. The DeBERTa output failed validation (F1 = 0.05–0.51 across constructs); LLM-as-classifier is the methodologically standard upgrade path for cases where zero-shot NLI is inadequate for subjective psychological construct measurement.

**Citation precedent for examiners.**
- Gilardi, F., Alizadeh, M., & Kubli, M. (2023). ChatGPT outperforms crowd workers for text-annotation tasks. *PNAS*, 120(30), e2305016120.
- Ziems, C., Held, W., Shaikh, O., et al. (2024). Can large language models transform computational social science? *Computational Linguistics*, 50(1), 237–291.

Both papers establish LLM annotation as a defensible approach in computational social science when (a) a coding rubric is well-specified and (b) results are validated against human-coded gold standard. Both conditions are met by your existing pipeline.

**Method overview.**
1. For each comment in `comments_clean.csv`, send a prompt to the LLM API containing your coding rubric and the comment text.
2. The LLM returns a structured JSON response with probability scores (0.0–1.0) and binary judgments per construct.
3. Save the new scores to `comments_scored_llm.csv`.
4. Validate against your existing 200-comment human-coded reference set (`validation_two_coders.csv`); compute F1 of LLM-vs-human per construct.
5. If F1 ≥ 0.65 on focal constructs, proceed to Notebook 03 (mixed-effects regression).

**API choice.** This notebook defaults to **Claude (Anthropic API)** because (a) Claude tends to be slightly more accurate on subjective construct classification, (b) the per-token cost is comparable to OpenAI, and (c) you're already in the Anthropic ecosystem. Switching to OpenAI requires changing two lines (clearly marked).

**Cost estimate.** Roughly $5–25 total for all 4,500 comments depending on model:
- Claude Haiku 4.5: ~$1–3
- Claude Sonnet 4.6: ~$15–25
- OpenAI GPT-4o-mini: ~$2–5
- OpenAI GPT-4o: ~$30–50

**Recommendation:** Start with **Claude Sonnet 4.6** for the dissertation — the cost is small relative to the project's stakes, and accuracy matters more than speed.

**Runtime.** ~30–60 minutes for 4,500 comments, depending on model and API throughput.


## 1. Setup — install dependencies and configure API key

Install the Anthropic SDK from a terminal (or use `%pip install` in a notebook cell):

```bash
pip install anthropic tqdm
```

Then set your Anthropic API key as an environment variable. **Do not paste your key into the notebook directly** (it will get committed if you ever share the notebook). Instead:

In your terminal (Mac):
```bash
export ANTHROPIC_API_KEY="sk-ant-..."
```

To make it persistent, add that line to your `~/.zshrc` (Mac) and run `source ~/.zshrc`. Get an API key at: https://console.anthropic.com/

**Verifying the key works:** the cell below will fail with a clear error if the key isn't set or is invalid. We do this check upfront so you don't waste time scoring 4,500 comments with a broken connection.


In [ ]:
import os
import json
import time
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from anthropic import Anthropic

# Will read ANTHROPIC_API_KEY from environment automatically
client = Anthropic()

# Quick test call to verify the API key works
try:
    test = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=20,
        messages=[{"role":"user","content":"Reply with the single word: OK"}]
    )
    print(f"API connection OK. Test response: {test.content[0].text.strip()}")
except Exception as e:
    raise RuntimeError(f"API connection failed: {e}\n\nCheck that ANTHROPIC_API_KEY is set in your environment.")


## 2. Load the cleaned corpus

Same input as Notebook 02 — the cleaned 4,500-comment Discussion-only sample. The classification overwrites no existing files; we'll write a new `comments_scored_llm.csv` to keep the DeBERTa scores intact for comparison.


In [ ]:
df = pd.read_csv("comments_clean.csv")

# Apply Option 1 filters (Discussion-only, drop low-volume influencers)
DROP = ["Samantha March","Amanda Z","Lydia Elise Millen","Victoria Magrath"]
df = df[df["subreddit_stratum"]=="discussion"].copy()
df = df[~df["matched_influencer"].isin(DROP)].copy()
df = df.reset_index(drop=True)
print(f"Comments to classify: {len(df):,}")
print(f"Tier balance: {df['influencer_tier'].value_counts().to_dict()}")


## 3. The classification prompt

The prompt is the methodological heart of this approach. It functions as a *coding manual* for the LLM, equivalent to what we'd give a human research assistant. Three principles guide the design:

1. **Specify the construct rubric in plain language**, with positive and negative examples for each construct, mirroring the rubric you used in your manual validation pass.
2. **Anchor the judgment to the focal influencer** so the LLM doesn't get confused when comments mention multiple people.
3. **Force structured output** (JSON) so the response is mechanically parseable rather than free-form prose.

The rubric below is the same one you applied during your own manual coding (and the same one Claude used as the second coder), refined slightly based on the disagreements we identified in the two-coder analysis.


In [ ]:
SYSTEM_PROMPT = '''You are a careful research coder for a Master's dissertation on social media influencer marketing. Your task is to read a Reddit comment and judge whether it expresses each of four psychological constructs toward a specific focal influencer.

Apply each construct STRICTLY to the focal influencer named below — not to other people mentioned in the comment.

THE FOUR CONSTRUCTS

1. BENIGN ENVY — The commenter expresses upward admiration toward the focal influencer as a person, with an aspirational or identification-oriented tone. They want to BE LIKE the influencer, treat them as a role model, find them inspiring, or admire their style/skill/qualities in an aspirational way.
   POSITIVE examples: "she's goals 😍", "I want to dress like her", "her glow up is so inspiring", "she's everything I want to be", "Hannah does insanely thorough reviews, I like her a lot", "Lauren Mae is great at describing eyeshadows".
   NEGATIVE examples: "this product is great" (about product, not person), "she's nice" (no aspiration), generic mention with no admiration, hostile/sarcastic comments.

2. MALICIOUS ENVY — The commenter expresses hostility, contempt, sarcasm, mockery, accusations of fakeness, dismissiveness, or bitter resentment toward the focal influencer as a person.
   POSITIVE examples: "so fake", "must be nice 🙄", "she's such a fraud", "Jaclyn Hill 2.0 with all her bullshit", "she's exhausting", explicit critiques of personality.
   NEGATIVE examples: criticism of a product (not the person), factual disagreement, neutral observation, comments hostile to someone other than the focal influencer.

3. PSI (PARASOCIAL INTERACTION) — The commenter writes about the focal influencer as if they personally know them — uses nicknames, defends them from criticism, expresses warm familiarity, treats them like a friend.
   POSITIVE examples: "we love her", "leave my girl alone", "Alix would never", "I feel like I grew up with her", "feels like chatting with a friend", defending the influencer in any way.
   NEGATIVE examples: general approval without closeness ("her work is good"), product discussion, hostile remarks.

4. PURCHASE INTENT — The commenter expresses urgent desire to buy a specific product (often in reaction to the influencer's recommendation).
   POSITIVE examples: "where is the link", "take my money", "adding to cart immediately", "I need this RIGHT NOW", "Julia Adams sold me on this", "I get the urge to try them".
   NEGATIVE examples: discussing past purchases without present urge, browsing without intent, anti-consumption remarks.

OUTPUT FORMAT

Return a single JSON object with this exact structure and no other text:

{
  "benign_envy":     {"score": <float 0-1>, "binary": <0 or 1>, "reason": "<one sentence>"},
  "malicious_envy":  {"score": <float 0-1>, "binary": <0 or 1>, "reason": "<one sentence>"},
  "psi":             {"score": <float 0-1>, "binary": <0 or 1>, "reason": "<one sentence>"},
  "purchase_intent": {"score": <float 0-1>, "binary": <0 or 1>, "reason": "<one sentence>"}
}

The "score" reflects how strongly the comment expresses the construct (0.0 = clearly not expressed, 1.0 = unambiguous expression). The "binary" is your 0/1 judgment at threshold 0.5. The "reason" is a brief justification.'''


def build_user_message(focal_influencer: str, body: str) -> str:
    return f"FOCAL INFLUENCER: {focal_influencer}\n\nCOMMENT:\n{body}"


## 4. Smoke test on 5 comments

Before scoring all 4,500 comments, the prompt is verified on a small subset. Five random comments are pulled and both the scores and the LLM's reasoning are inspected. The reasoning field serves as a transparency check — it exposes whether the LLM is interpreting each construct correctly.


In [ ]:
def classify_one(focal_influencer: str, body: str,
                 model: str = "claude-sonnet-4-6", retries: int = 3):
    """Send one comment to the LLM and parse the JSON response."""
    user_msg = build_user_message(focal_influencer, body)
    last_err = None
    for attempt in range(retries):
        try:
            resp = client.messages.create(
                model=model,
                max_tokens=600,
                system=SYSTEM_PROMPT,
                messages=[{"role":"user","content":user_msg}],
            )
            text = resp.content[0].text.strip()
            # Strip any code-fence wrappers the model occasionally adds
            if text.startswith("```"):
                text = text.split("```")[1]
                if text.startswith("json"):
                    text = text[4:].strip()
            return json.loads(text)
        except (json.JSONDecodeError, IndexError) as e:
            last_err = f"Parse error: {e}"
            time.sleep(1 + attempt)
        except Exception as e:
            last_err = f"API error: {e}"
            time.sleep(2 ** attempt)
    raise RuntimeError(f"Failed after {retries} attempts: {last_err}")


# Run smoke test
sample = df.sample(5, random_state=42)
print("SMOKE TEST — 5 comments\n" + "="*70)
for _, row in sample.iterrows():
    body_preview = str(row['body'])[:200].replace('\n',' ')
    print(f"\n[{row['matched_influencer']}] {body_preview}")
    result = classify_one(row['matched_influencer'], row['body'])
    for k, v in result.items():
        print(f"  {k:<18} score={v['score']:.2f} binary={v['binary']}  — {v['reason']}")


**Pause and read those samples carefully.** The five spot-checks are your sanity check on the prompt. Look for:

- Does the LLM correctly identify aspirational comments as `benign_envy`?
- Does it correctly distinguish hostility *toward the influencer* from criticism of a product or third party?
- Are PSI judgments grounded in expressed closeness (not just positive sentiment)?
- Is purchase_intent flagged only on present urgency, not past purchases?

If any of these look wrong on the smoke test, edit the rubric examples in the SYSTEM_PROMPT (cell above) and re-run the smoke test. Don't proceed to the full classification until the smoke test looks consistently correct — fixing the prompt at this stage is free; fixing it after spending an hour scoring 4,500 comments is not.


## 5. Full classification with checkpoint and resume

The same resilience pattern as the DeBERTa notebook: process in small batches, save a checkpoint to disk every 50 comments, skip already-processed comments on re-run. Interrupting the run (Ctrl+C) is safe — at most the most recent batch is lost.

**API rate limits.** Anthropic's default rate limits allow comfortably more requests than this notebook sends. If a 429 error is returned, the retry logic in `classify_one` waits and retries with exponential backoff.


In [ ]:
CHECKPOINT_PATH = "comments_scored_llm_partial.csv"
SAVE_EVERY = 50      # comments
MODEL = "claude-sonnet-4-6"

# Resume support: skip comments already in the partial checkpoint
done_ids = set()
if os.path.exists(CHECKPOINT_PATH):
    prior = pd.read_csv(CHECKPOINT_PATH)
    done_ids = set(prior["id"])
    print(f"Resuming — {len(done_ids):,} comments already scored.")

todo = df[~df["id"].isin(done_ids)].reset_index(drop=True)
print(f"Comments to score this run: {len(todo):,}")

# Score, with periodic checkpoint
scored_rows = []
buffer = []
for i, row in tqdm(todo.iterrows(), total=len(todo), desc="LLM classifying"):
    try:
        out = classify_one(row['matched_influencer'], row['body'], model=MODEL)
        scored_rows.append({
            "id": row["id"],
            "matched_influencer": row["matched_influencer"],
            "influencer_tier": row["influencer_tier"],
            "subreddit": row["subreddit"],
            "subreddit_stratum": row["subreddit_stratum"],
            "tier_mega": row.get("tier_mega", int(row["influencer_tier"]=="mega")),
            "sub_snark": row.get("sub_snark", int(row["subreddit_stratum"]=="snark")),
            "body": row["body"],
            "body_len": row.get("body_len", len(str(row["body"]).split())),
            "month_window": row.get("month_window", ""),
            "permalink": row.get("permalink", ""),
            "benign_envy":     out["benign_envy"]["score"],
            "malicious_envy":  out["malicious_envy"]["score"],
            "psi":             out["psi"]["score"],
            "purchase_intent": out["purchase_intent"]["score"],
            "benign_envy_binary":     out["benign_envy"]["binary"],
            "malicious_envy_binary":  out["malicious_envy"]["binary"],
            "psi_binary":             out["psi"]["binary"],
            "purchase_intent_binary": out["purchase_intent"]["binary"],
        })
        buffer.append(scored_rows[-1])

        if len(buffer) >= SAVE_EVERY:
            new_df = pd.DataFrame(scored_rows)
            if os.path.exists(CHECKPOINT_PATH):
                old = pd.read_csv(CHECKPOINT_PATH)
                full = pd.concat([old, new_df], ignore_index=True).drop_duplicates("id")
            else:
                full = new_df
            full.to_csv(CHECKPOINT_PATH, index=False)
            scored_rows = []
            buffer = []
    except Exception as e:
        print(f"  Skipped id={row['id']}: {e}")

# Final flush
if scored_rows:
    new_df = pd.DataFrame(scored_rows)
    if os.path.exists(CHECKPOINT_PATH):
        old = pd.read_csv(CHECKPOINT_PATH)
        full = pd.concat([old, new_df], ignore_index=True).drop_duplicates("id")
    else:
        full = new_df
    full.to_csv(CHECKPOINT_PATH, index=False)

print(f"\nDone. Checkpoint at {CHECKPOINT_PATH}")


## 6. Score distributions — first sanity check

Same diagnostic as Notebook 02 cell 7. Histograms of the four LLM-generated scores. We want to see realistic distributions (long-tailed, not stuck at extremes), with most comments scoring near 0 on any given construct and a smaller tail of strong-signal comments scoring high.


In [ ]:
import matplotlib.pyplot as plt

scored_llm = pd.read_csv(CHECKPOINT_PATH)
print(f"Total LLM-scored comments: {len(scored_llm):,}")
print("\nScore summary:")
print(scored_llm[["benign_envy","malicious_envy","psi","purchase_intent"]].describe().round(3))

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, name in zip(axes.flat, ["benign_envy","malicious_envy","psi","purchase_intent"]):
    ax.hist(scored_llm[name], bins=50)
    ax.set_title(name)
    ax.set_xlabel("LLM score (0-1)")
plt.suptitle("Distribution of LLM-classifier scores", y=1.02)
plt.tight_layout()
plt.show()


## 7. Validate against the 200-sample human reference

The decisive step. Compute F1 of the LLM scores against your existing human-coded validation set (`validation_two_coders.csv`). If F1 ≥ 0.65 per focal construct, the LLM measurement is defensible for confirmatory hypothesis testing. If not, we'll need to revisit the prompt or escalate the model.


In [ ]:
from sklearn.metrics import precision_recall_fscore_support, cohen_kappa_score

ref = pd.read_csv("validation_two_coders.csv")
joined = ref.merge(scored_llm[["id","benign_envy","malicious_envy","psi","purchase_intent",
                                "benign_envy_binary","malicious_envy_binary","psi_binary","purchase_intent_binary"]],
                    on="id", suffixes=("","_llm"))
print(f"Validation rows merged: {len(joined)}")

constructs = ["benign_envy","malicious_envy","psi","purchase_intent"]
print("\n" + "="*78)
print("LLM-classifier vs human reference (your codes + Claude's codes)")
print("="*78)
print(f"{'Construct':<20}{'vs USER F1':<14}{'vs CLAUDE-coder F1':<22}")
print("-"*78)
for c in constructs:
    yp = joined[f"{c}_binary"].astype(int)  # LLM binary judgment
    yu = joined[f"human_{c}"].astype(int)
    yc = joined[f"claude_{c}"].astype(int)
    _,_,fu,_ = precision_recall_fscore_support(yu, yp, average="binary", zero_division=0)
    _,_,fc,_ = precision_recall_fscore_support(yc, yp, average="binary", zero_division=0)
    print(f"{c:<20}{fu:<14.3f}{fc:<22.3f}")

# Decision summary
print("\n" + "="*78)
print("VALIDITY DECISION (using max of vs-USER and vs-CLAUDE F1)")
print("="*78)
for c in constructs:
    yp = joined[f"{c}_binary"].astype(int)
    fu = precision_recall_fscore_support(joined[f"human_{c}"].astype(int), yp, average="binary", zero_division=0)[2]
    fc = precision_recall_fscore_support(joined[f"claude_{c}"].astype(int), yp, average="binary", zero_division=0)[2]
    f = max(fu, fc)
    if f >= 0.75:   verdict = "STRONG  — confirmatory use OK"
    elif f >= 0.65: verdict = "ACCEPTABLE — use, report F1"
    elif f >= 0.50: verdict = "MARGINAL — exploratory only"
    else:           verdict = "POOR — revisit prompt or escalate model"
    print(f"  {c:<22} F1={f:.3f}  →  {verdict}")


## 8. Save the final scored corpus and decide

If validation passes, the LLM-scored corpus is saved as `comments_scored_llm.csv` — the file loaded by Notebook 03 (mixed-effects regressions). The original `comments_scored.csv` (DeBERTa) is retained intact to allow comparison of the two measurement approaches in the Methods chapter (a convergent-validity check).


In [ ]:
scored_llm.to_csv("comments_scored_llm.csv", index=False)
print(f"Saved comments_scored_llm.csv ({len(scored_llm):,} rows)")
print("\nReady for Notebook 03 (mixed-effects regressions) if validation passed.")
